In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tars06/cat-img/cat2.jpg


In [3]:
import torch
import torch.nn as nn

class AdaLNZero(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.norm = nn.LayerNorm(hidden_dim, elementwise_affine = False)

        self.time_comb_proj = nn.Linear(hidden_dim, 3*hidden_dim)

        nn.init.zeros_(self.time_comb_proj.weight)
        nn.init.zeros_(self.time_comb_proj.bias)

    def forward(self, x, t_emb):
        modulation = self.time_comb_proj(t_emb)

        gamma, beta, alpha = torch.chunk(modulation, 3, dim = -1)
        x_norm = self.norm(x)
        x_modulated = x_norm * (1+ gamma.unsqueeze(1)) + beta.unsqueeze(1)

        return x_modulated, alpha

In [4]:
adaln_layer = AdaLNZero(hidden_dim=256)
mock_patches = torch.randn(1,1024, 256)

mock_time_emb = torch.randn(1,256)
modulated_tokens, scale_alpha = adaln_layer(mock_patches, mock_time_emb)

print("Input patches shape : ", mock_patches.shape)
print("modulated output shape : ", modulated_tokens.shape)
print("Residual scale alpha : ", scale_alpha.shape)
print("\n--- ZERO-INITIALIZATION CHECK ---")
print("First 5 values of alpha (Should be exactly 0.0 due to zero-init):")
print(scale_alpha[0, :5].detach().numpy())

Input patches shape :  torch.Size([1, 1024, 256])
modulated output shape :  torch.Size([1, 1024, 256])
Residual scale alpha :  torch.Size([1, 256])

--- ZERO-INITIALIZATION CHECK ---
First 5 values of alpha (Should be exactly 0.0 due to zero-init):
[0. 0. 0. 0. 0.]


In [11]:
import torch
import torch.nn as nn
import math

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, hidden_dim = 256, num_heads=4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        assert self.head_dim * num_heads == hidden_dim, "hidden_dim must be div by num_head"

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        B, N, C = x.shape

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        Q = Q.view(B, N, self.num_heads, self.head_dim).permute(0,2,1,3)
        K = K.view(B, N, self.num_heads, self.head_dim).permute(0,2,1,3)
        V = V.view(B, N, self.num_heads, self.head_dim).permute(0,2,1,3)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_weights = torch.softmax(scores, dim =-1)

        context = torch.matmul(attn_weights, V)

        context = context.permute(0,2,1,3).contiguous()
        context = context.view(B, N, C)

        return self.out_proj(context)
        

In [12]:
attention_layer = MultiHeadSelfAttention(hidden_dim = 256, num_heads=4)

mock_modulated_tokens = torch.randn(1,1024,256)

attention_output = attention_layer(mock_modulated_tokens)

print("Input tokens shape: ", mock_modulated_tokens.shape)
print("output tokens shape", attention_output.shape)

Input tokens shape:  torch.Size([1, 1024, 256])
output tokens shape torch.Size([1, 1024, 256])
